# LAB 1 — Query Rewriting: Reduzindo o Vocabulary Gap
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Implementar e comparar 3 técnicas de query rewriting em queries jurídicas reais,
medindo o impacto no retrieval antes e depois da reformulação.

**Entregáveis:**
- Pipeline de query rewriting funcional com 3 técnicas
- Tabela comparativa: query original vs reformulada para 5 queries do baseline
- Análise: qual técnica gerou maior diversidade semântica?

**Tempo estimado:** 40 minutos  
**Pré-requisito:** Aula 2 concluída — OpenSearch com corpus indexado


## ⚙️ Passo 1 — Instalação e Imports

In [ ]:
# Instalar dependências
%pip install -q langchain langchain-community langchain-openai langchain-ollama langchain-groq groq openai python-dotenv sentence-transformers opensearch-py pandas numpy
print("✅ Instalação concluída")

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from typing import List, Dict, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Carrega ~/mba-rag/.env (ou .env local) se existir
load_dotenv()

# ── Configurações Groq (primário) ─────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# ── Configurações Ollama (fallback local) ─────────────────────────
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "llama3.2:3b")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3")

# ── Configurações OpenSearch ──────────────────────────────────────
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
INDEX_NAME      = os.getenv("INDEX_NAME", "corpus_juridico")

print("✅ Configurações carregadas:")
print(f"   LLM primário:   Groq → {GROQ_LLM_MODEL}  ({'API key OK' if GROQ_API_KEY else 'SEM key → cairá no Ollama'})")
print(f"   LLM fallback:   Ollama → {OLLAMA_LLM_MODEL}  ({OLLAMA_BASE_URL})")
print(f"   Embeddings:     Ollama → {OLLAMA_EMBED_MODEL} (fallback: HuggingFace BAAI/bge-m3)")
print(f"   OpenSearch:     {OPENSEARCH_HOST}:{OPENSEARCH_PORT} | índice: {INDEX_NAME}")

## 🔌 Passo 2 — Conectar ao LLM e OpenSearch

In [ ]:
from openai import OpenAI

# ─────────────────────────────────────────────────────────────────
# Estratégia: Groq primário (llama-3.1-8b-instant) + Ollama fallback
# Ambos expõem API OpenAI-compatible, então usamos um único cliente.
# ─────────────────────────────────────────────────────────────────

llm_client = None
MODEL_NAME = None
LLM_PROVIDER = None

# Tentativa 1 — Groq (nuvem, baixa latência)
if GROQ_API_KEY:
    try:
        candidate = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)
        _ = candidate.chat.completions.create(
            model=GROQ_LLM_MODEL,
            messages=[{"role": "user", "content": "ok"}],
            max_tokens=2, temperature=0.0,
        )
        llm_client, MODEL_NAME, LLM_PROVIDER = candidate, GROQ_LLM_MODEL, "groq"
        print(f"✅ LLM via Groq | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"⚠️  Groq falhou ({e}); caindo para Ollama local…")

# Tentativa 2 — Ollama (fallback local, API OpenAI-compatible em /v1)
if llm_client is None:
    try:
        candidate = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
        _ = candidate.chat.completions.create(
            model=OLLAMA_LLM_MODEL,
            messages=[{"role": "user", "content": "ok"}],
            max_tokens=2, temperature=0.0,
        )
        llm_client, MODEL_NAME, LLM_PROVIDER = candidate, OLLAMA_LLM_MODEL, "ollama"
        print(f"✅ LLM via Ollama (fallback) | {OLLAMA_BASE_URL} | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"❌ Nenhum provedor LLM disponível: {e}")
        print("   Configure GROQ_API_KEY no .env ou rode `ollama serve` localmente.")


def llm_call(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
    """Chama o LLM ativo (Groq ou Ollama) e retorna o texto gerado."""
    if llm_client is None:
        return ""
    try:
        resp = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️  LLM ({LLM_PROVIDER}) falhou: {e}")
        return ""


if llm_client is not None:
    test = llm_call("Responda em uma palavra: qual é a capital do Brasil?")
    print(f"   Smoke test → '{test}' (esperado: Brasília)")

In [ ]:
from opensearchpy import OpenSearch

# Conectar ao OpenSearch
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=("admin", os.getenv("OPENSEARCH_PASSWORD", "admin")),
    use_ssl=False,
    verify_certs=False
)

try:
    info = os_client.info()
    print(f"✅ OpenSearch conectado! Versão: {info['version']['number']}")
    count = os_client.count(index=INDEX_NAME)['count']
    print(f"   Documentos no índice '{INDEX_NAME}': {count}")
except Exception as e:
    print(f"⚠️  OpenSearch não disponível: {e}")
    print("   O lab continuará em modo simulado para fins didáticos")

## 📋 Passo 3 — Carregar Queries do Baseline

In [ ]:
# Carregar dataset com queries
DATASET_PATH = "../datasets/corpus_juridico_aula3.json"

try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    queries_baseline = dataset["queries_baseline"]
    print(f"✅ Dataset carregado: {len(queries_baseline)} queries")
except Exception:
    # Fallback: queries hardcoded para demonstração
    queries_baseline = [
        {"id": "q_001", "query_original": "Podem prender alguém sem mandado?"},
        {"id": "q_002", "query_original": "Policial pode ler mensagens do celular do suspeito?"},
        {"id": "q_003", "query_original": "O suspeito é obrigado a falar na delegacia?"},
        {"id": "q_004", "query_original": "O advogado pode ver o inquérito policial?"},
        {"id": "q_005", "query_original": "Como provar que o réu tinha intenção de matar?"},
    ]
    print("⚠️  Usando queries hardcoded (dataset não encontrado)")

print()
print("📋 Queries do baseline:")
for q in queries_baseline[:5]:
    print(f"  [{q['id']}] {q['query_original']}")

## 🔄 Passo 4 — Implementar as 3 Técnicas de Rewriting

In [ ]:
# ── TÉCNICA 1: Paraphrase Rewriting ──────────────────────────────

def paraphrase_rewriting(query: str, n: int = 3) -> List[str]:
    """Gera N reformulações técnicas da query jurídica."""
    prompt = f"""Você é especialista em direito processual penal brasileiro.
Reformule a query abaixo em {n} variações usando terminologia técnica jurídica.
Mantenha o mesmo significado, mas use linguagem técnica de documentos jurídicos.

Query original: {query}

Retorne APENAS as {n} variações, uma por linha, sem numeração ou introdução."""
    
    result = llm_call(prompt, temperature=0.5)
    if not result:
        return [query]  # fallback
    
    lines = [l.strip() for l in result.split('\n') if l.strip()]
    return lines[:n] if lines else [query]


# ── TÉCNICA 2: HyDE-Lite ─────────────────────────────────────────

def hyde_lite(query: str) -> str:
    """Gera parágrafo hipotético técnico para uso como query de busca."""
    prompt = f"""Você é redator jurídico especializado em processo penal brasileiro.
Escreva um parágrafo curto (4-6 linhas) que seria encontrado em acórdão ou código 
e que responderia diretamente à pergunta. Use linguagem técnica formal.

Pergunta: {query}

Escreva APENAS o parágrafo, sem título."""
    
    result = llm_call(prompt, temperature=0.2, max_tokens=300)
    return result if result else query


# ── TÉCNICA 3: Step-Back ─────────────────────────────────────────

def step_back(query: str) -> str:
    """Abstrai a query para nível conceitual mais geral."""
    prompt = f"""Você é especialista em direito processual penal.
Dada a pergunta específica abaixo, formule uma pergunta MAIS GERAL 
que abrange o conceito jurídico por trás dela.

Pergunta específica: {query}

Retorne APENAS a pergunta geral, sem explicação."""
    
    result = llm_call(prompt, temperature=0.1, max_tokens=100)
    return result if result else query

print("✅ Três técnicas de rewriting implementadas:")
print("   1. paraphrase_rewriting(query, n=3)")
print("   2. hyde_lite(query)")
print("   3. step_back(query)")

## 🧪 Passo 5 — Executar Rewriting nas 5 Queries

In [ ]:
# Processar todas as queries do baseline
print("⏳ Processando queries com as 3 técnicas de rewriting...")
print("   (Isso pode levar 1-2 minutos dependendo do LLM)")
print()

resultados_rewriting = []

for q in queries_baseline[:5]:
    qid = q["id"]
    original = q["query_original"]
    print(f"Processando {qid}: '{original}'")
    
    t0 = time.time()
    
    # Aplicar as 3 técnicas
    paraphrases = paraphrase_rewriting(original, n=2)
    hyde_doc    = hyde_lite(original)
    stepback_q  = step_back(original)
    
    elapsed = time.time() - t0
    
    resultados_rewriting.append({
        "id": qid,
        "original": original,
        "paraphrase_1": paraphrases[0] if len(paraphrases) > 0 else original,
        "paraphrase_2": paraphrases[1] if len(paraphrases) > 1 else original,
        "hyde_lite": hyde_doc[:200] + "..." if len(hyde_doc) > 200 else hyde_doc,
        "step_back": stepback_q,
        "tempo_s": round(elapsed, 2)
    })
    
    print(f"  ✅ Concluído em {elapsed:.1f}s")

print()
print(f"✅ {len(resultados_rewriting)} queries processadas!")

## 📊 Passo 6 — Tabela Comparativa

In [ ]:
# Exibir tabela comparativa
print("📊 TABELA COMPARATIVA: Query Original vs Reformulações")
print("=" * 80)
print()

for r in resultados_rewriting:
    print(f"━━━ [{r['id']}] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"ORIGINAL:     {r['original']}")
    print(f"PARAPHRASE 1: {r['paraphrase_1']}")
    print(f"PARAPHRASE 2: {r['paraphrase_2']}")
    print(f"HYDE-LITE:    {r['hyde_lite'][:120]}...")
    print(f"STEP-BACK:    {r['step_back']}")
    print(f"Tempo:        {r['tempo_s']}s")
    print()

# Salvar como CSV para análise
df = pd.DataFrame(resultados_rewriting)
df.to_csv("/tmp/rewriting_comparativo.csv", index=False, encoding="utf-8")
print("✅ Resultados salvos em /tmp/rewriting_comparativo.csv")

## 🔎 Passo 7 — Testar Impacto no Retrieval

In [ ]:
def buscar_opensearch(query: str, top_k: int = 5) -> List[Dict]:
    """Realiza busca textual + vetorial no OpenSearch."""
    try:
        response = os_client.search(
            index=INDEX_NAME,
            body={
                "query": {
                    "multi_match": {
                        "query": query,
                        "fields": ["texto^2", "ementa", "palavras_chave"],
                        "type": "best_fields"
                    }
                },
                "size": top_k
            }
        )
        hits = response["hits"]["hits"]
        return [
            {
                "id": h["_id"],
                "score": round(h["_score"], 3),
                "texto": h["_source"].get("texto", "")[:200] + "...",
                "fonte": h["_source"].get("numero", h["_id"])
            }
            for h in hits
        ]
    except Exception as e:
        # Modo simulado
        return [
            {"id": "doc_sim_1", "score": 0.95, "texto": "Resultado simulado 1...", "fonte": "CF/88 Art. 5º"},
            {"id": "doc_sim_2", "score": 0.88, "texto": "Resultado simulado 2...", "fonte": "CPP Art. 302"},
        ]

# Testar com a primeira query
query_teste = resultados_rewriting[0]
print(f"🔎 Testando retrieval para: [{query_teste['id']}]")
print()

print("QUERY ORIGINAL:")
res_original = buscar_opensearch(query_teste["original"])
for r in res_original:
    print(f"  [{r['score']:.3f}] {r['fonte']}")

print()
print("QUERY STEP-BACK (mais abrangente):")
res_stepback = buscar_opensearch(query_teste["step_back"])
for r in res_stepback:
    print(f"  [{r['score']:.3f}] {r['fonte']}")

# Calcular overlap
ids_original = {r["id"] for r in res_original}
ids_stepback = {r["id"] for r in res_stepback}
overlap = len(ids_original & ids_stepback) / max(len(ids_original), 1)
novos = len(ids_stepback - ids_original)

print()
print(f"📊 Análise de diversidade:")
print(f"   Overlap entre técnicas: {overlap:.0%}")
print(f"   Documentos novos recuperados pelo step-back: {novos}")

## ✅ Passo 8 — Reflexão e Análise

In [ ]:
print("📝 ANÁLISE FINAL — Query Rewriting")
print("=" * 60)
print()
print("Perguntas para reflexão:")
print()
print("1️⃣  Qual técnica gerou queries mais diferentes da original?")
print("    → Compare lexicalmente: HyDE-Lite tende a ser mais longa e técnica")
print()
print("2️⃣  Para queries jurídicas coloquiais, qual técnica é mais eficaz?")
print("    → Step-back é eficaz quando há forte jargão operacional vs formal")
print()
print("3️⃣  O HyDE-Lite pode causar problemas. Quais?")
print("    → RISCO: Se o LLM alucinar no parágrafo hipotético,")
print("       a busca vai recuperar documentos irrelevantes.")
print("       Em ambiente jurídico, isso é especialmente crítico!")
print()
print("4️⃣  Qual o custo de cada técnica?")
print("    Paraphrase: ~100 tokens | HyDE-Lite: ~200 tokens | Step-back: ~50 tokens")
print()
print("✅ LAB 1 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] 3 técnicas implementadas e executadas")
print("   [ ] Tabela comparativa gerada para 5 queries")
print("   [ ] Teste de retrieval com query original vs reformulada")
print("   [ ] Arquivo CSV salvo em /tmp/rewriting_comparativo.csv")
print("   [ ] Reflexão respondida nos comentários abaixo")

## 📝 Espaço para Reflexão

*Complete aqui suas observações:*

**Qual técnica apresentou maior diversidade semântica?**
→ *[sua resposta]*

**Para o domínio jurídico policial (jargão operacional), qual técnica recomendaria?**
→ *[sua resposta]*

**Algum caso onde o HyDE-Lite gerou resultado inadequado?**
→ *[sua resposta]*
